In [ ]:
rankings = {
    'louvain': {
        '0.2': ['Eigenvector', 'Triangle Participation', 'Expansion', 'Clustering Coefficient', 'E Out'],
        '0.3': ['Eigenvector', 'Triangle Participation', 'Expansion', 'Clustering Coefficient', 'E Out'],
        '0.4': ['Triangle Participation','Eigenvector', 'Expansion', 'Clustering Coefficient', 'Betweenness']
    },
    'lpa':{
        '0.2': ['Eigenvector', 'Expansion', 'Triangle Participation','E Out', 'Clustering Coefficient'],
        '0.3': ['Eigenvector','Clustering Coefficient', 'Expansion','Triangle Participation', 'E Out']
    }
}

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import os
from pathlib import Path

# ============================================
# RANKINGS DE REFERÊNCIA (Top 5 por importance_mean)
# ============================================

rankings = {
    'louvain': {
        '0_2': ['Eigenvector', 'Triangle Participation', 'Expansion', 'Clustering Coefficient', 'E Out'],
        '0_3': ['Eigenvector', 'Triangle Participation', 'Expansion', 'Clustering Coefficient', 'E Out'],
        '0_4': ['Triangle Participation', 'Eigenvector', 'Expansion', 'Clustering Coefficient', 'Betweenness']
    },
    'lpa': {
        '0_2': ['Eigenvector', 'Expansion', 'Triangle Participation', 'E Out', 'Clustering Coefficient'],
        '0_3': ['Eigenvector', 'Clustering Coefficient', 'Expansion', 'Triangle Participation', 'E Out']
    }
}

# ============================================
# FUNÇÃO PARA CARREGAR ARQUIVO ALE
# ============================================

def load_ale_importance(file_path):
    """
    Carrega o arquivo CSV de importância ALE.
    """
    if not os.path.exists(file_path):
        return None
    
    df = pd.read_csv(file_path)
    
    if 'feature' not in df.columns or 'importance_mean' not in df.columns:
        return None
    
    return df

# ============================================
# FUNÇÃO PARA EXTRATIR TOP N FEATURES
# ============================================

def get_top_n_features(df, n=5):
    """Retorna as top N features por importance_mean."""
    if df is None or df.empty:
        return [], None
    
    df_sorted = df.sort_values('importance_mean', ascending=False)
    top_df = df_sorted[['feature', 'importance_mean']].head(n).copy()
    top_df['rank'] = range(1, len(top_df) + 1)
    top_list = top_df['feature'].tolist()
    
    return top_list, top_df

# ============================================
# FUNÇÃO PARA CALCULAR CORRELAÇÃO DE SPEARMAN
# ============================================

def compare_rankings(ranking_ref, ranking_ale):
    """
    Calcula a correlação de Spearman entre dois rankings.
    """
    common_features = set(ranking_ref) & set(ranking_ale)
    
    if len(common_features) < 2:
        return {
            'spearman': np.nan,
            'p_value': np.nan,
            'n_common': len(common_features),
            'status': '⚠️ Poucas features em comum'
        }
    
    ref_common = [f for f in ranking_ref if f in common_features]
    ale_common = [f for f in ranking_ale if f in common_features]
    
    ref_pos = {f: i for i, f in enumerate(ref_common)}
    ale_pos = {f: i for i, f in enumerate(ale_common)}
    
    ref_order = [ref_pos[f] for f in ref_common]
    ale_order = [ale_pos[f] for f in ref_common]
    
    corr, p_val = spearmanr(ref_order, ale_order)
    
    if abs(corr) >= 0.8:
        status = '✅ Alta concordância'
    elif abs(corr) >= 0.6:
        status = '⚠️ Concordância moderada'
    else:
        status = '❌ Baixa concordância'
    
    return {
        'spearman': corr,
        'p_value': p_val,
        'n_common': len(common_features),
        'status': status
    }

# ============================================
# FUNÇÃO PARA IMPRIMIR RESULTADOS
# ============================================

def print_comparison_results(algorithm, mu, ref_ranking, ale_ranking, ale_df, result):
    """Exibe os resultados da comparação."""
    print("\n" + "=" * 70)
    print(f"📊 {algorithm.upper()} - μ = {mu}")
    print("=" * 70)
    
    print("\n🏷️  Ranking de Referência (Top 5):")
    for i, feat in enumerate(ref_ranking, 1):
        print(f"   {i}. {feat}")
    
    print("\n📈 Ranking ALE (Top 5 por importance_mean):")
    for _, row in ale_df.iterrows():
        print(f"   {int(row['rank'])}. {row['feature']} ({row['importance_mean']:.4f})")
    
    print("\n📊 Correlação de Spearman:")
    print(f"   Coeficiente: {result['spearman']:.3f}")
    print(f"   p-valor: {result['p_value']:.4f}")
    print(f"   Features em comum: {result['n_common']}/5")
    print(f"   Status: {result['status']}")

# ============================================
# FUNÇÃO PARA COMPARAR MÚLTIPLOS ARQUIVOS
# ============================================

def compare_all_files(base_path, configs):
    """
    Compara todos os arquivos ALE para as configurações especificadas.
    
    Args:
        base_path: caminho base para os arquivos CSV
        configs: lista de tuplas (algorithm, mu)
    """
    all_results = {}
    
    print("=" * 70)
    print("🔍 ANÁLISE DE CORRELAÇÃO DE SPEARMAN: ALE vs RANKING DE REFERÊNCIA")
    print("=" * 70)
    
    for algorithm, mu in configs:
        file_path = Path(base_path) / algorithm / f"mu_{mu}" / f"ale_feature_importance_{algorithm}_mu_{mu}.csv"
        
        # Verifica se o arquivo existe
        if not file_path.exists():
            print(f"\n⚠️ Arquivo não encontrado: {file_path}")
            continue
        
        # Verifica se o ranking de referência existe
        ref_ranking = rankings.get(algorithm, {}).get(mu, [])
        if not ref_ranking:
            print(f"\n⚠️ Ranking de referência não definido para {algorithm}, μ={mu}")
            continue
        
        # Carrega dados
        df = load_ale_importance(str(file_path))
        if df is None:
            print(f"\n⚠️ Erro ao carregar {file_path}")
            continue
        
        # Obtém Top 5 ALE
        ale_ranking, ale_df = get_top_n_features(df, n=5)
        
        # Compara rankings
        result = compare_rankings(ref_ranking, ale_ranking)
        
        # Armazena resultado
        key = (algorithm, mu)
        all_results[key] = {
            'result': result,
            'ale_ranking': ale_ranking,
            'ale_df': ale_df,
            'ref_ranking': ref_ranking,
            'df': df
        }
        
        # Exibe resultados
        print_comparison_results(algorithm, mu, ref_ranking, ale_ranking, ale_df, result)
    
    return all_results

# ============================================
# FUNÇÃO PARA CRIAR TABELA RESUMO
# ============================================

def create_summary_table(all_results):
    """Cria uma tabela resumo com todos os resultados."""
    data = []
    
    for (algo, mu), info in all_results.items():
        result = info['result']
        data.append({
            'Algoritmo': algo.upper(),
            'μ': mu,
            'Spearman ρ': result['spearman'],
            'p-valor': result['p_value'],
            'Features Comuns': result['n_common'],
            'Status': result['status']
        })
    
    df_summary = pd.DataFrame(data)
    return df_summary

# ============================================
# FUNÇÃO PARA CRIAR TABELA COMPARATIVA COMPLETA
# ============================================

def create_complete_table(all_results):
    """Cria uma tabela comparativa completa com rankings e correlações."""
    data = []
    
    for (algo, mu), info in all_results.items():
        result = info['result']
        
        # Formata rankings como string
        ale_str = " > ".join(info['ale_ranking'])
        ref_str = " > ".join(info['ref_ranking'])
        
        # Adiciona valores de importance_mean
        ale_values = [f"{row['feature']}({row['importance_mean']:.3f})" for _, row in info['ale_df'].iterrows()]
        ale_detail = " > ".join(ale_values)
        
        data.append({
            'Algoritmo': algo.upper(),
            'μ': mu,
            'Ranking ALE': ale_str,
            'Ranking Ref.': ref_str,
            'ALE (valores)': ale_detail,
            'Spearman ρ': result['spearman'],
            'p-valor': result['p_value'],
            'Status': result['status']
        })
    
    df_comp = pd.DataFrame(data)
    return df_comp

# ============================================
# FUNÇÃO PARA GERAR TABELA LaTeX
# ============================================

def generate_latex_table(df_summary):
    """Gera uma tabela LaTeX a partir do resumo."""
    latex_lines = []
    
    latex_lines.append("\\begin{table}[H]")
    latex_lines.append("\\centering")
    latex_lines.append("\\caption{Correlação de Spearman entre rankings ALE e de referência}")
    latex_lines.append("\\label{tab:spearman_ale_ranking}")
    latex_lines.append("\\begin{tabular}{lcccc}")
    latex_lines.append("\\hline")
    latex_lines.append("\\textbf{Algoritmo} & \\textbf{$\\mu$} & \\textbf{Spearman $\\rho$} & \\textbf{p-valor} & \\textbf{Status} \\\\")
    latex_lines.append("\\hline")
    
    for _, row in df_summary.iterrows():
        spearman = f"{row['Spearman ρ']:.3f}" if not pd.isna(row['Spearman ρ']) else "N/A"
        p_val = f"{row['p-valor']:.4f}" if not pd.isna(row['p-valor']) else "N/A"
        latex_lines.append(
            f"{row['Algoritmo']} & {row['μ']} & {spearman} & {p_val} & {row['Status']} \\\\"
        )
    
    latex_lines.append("\\hline")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# ============================================
# FUNÇÃO PARA GERAR TABELA LaTeX COMPLETA
# ============================================

def generate_latex_table_complete(df_comp):
    """Gera uma tabela LaTeX completa com rankings."""
    latex_lines = []
    
    latex_lines.append("\\begin{table}[H]")
    latex_lines.append("\\centering")
    latex_lines.append("\\caption{Comparação entre rankings ALE e de referência}")
    latex_lines.append("\\label{tab:ale_ranking_comparison}")
    latex_lines.append("\\begin{tabular}{lcllcc}")
    latex_lines.append("\\hline")
    latex_lines.append("\\textbf{Algoritmo} & \\textbf{$\\mu$} & \\textbf{Ranking ALE} & \\textbf{Ranking Ref.} & \\textbf{Spearman $\\rho$} & \\textbf{Status} \\\\")
    latex_lines.append("\\hline")
    
    for _, row in df_comp.iterrows():
        spearman = f"{row['Spearman ρ']:.3f}" if not pd.isna(row['Spearman ρ']) else "N/A"
        latex_lines.append(
            f"{row['Algoritmo']} & {row['μ']} & {row['Ranking ALE']} & {row['Ranking Ref.']} & {spearman} & {row['Status']} \\\\"
        )
    
    latex_lines.append("\\hline")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# ============================================
# EXECUÇÃO PRINCIPAL
# ============================================

if __name__ == "__main__":
    
    # ============================================
    # CONFIGURAÇÃO: Definir algoritmos e μ a comparar
    # ============================================
    
    # Configurações conforme os rankings disponíveis
    configs = [
        ('louvain', '0_2'),
        ('louvain', '0_3'),
        ('louvain', '0_4'),
        ('lpa', '0_2'),
        ('lpa', '0_3')
    ]
    
    # Caminho base
    base_path = "Resultados/ale_feature_importance"
    
    # ============================================
    # EXECUTAR COMPARAÇÃO
    # ============================================
    
    print("\n" + "=" * 70)
    print("📁 CONFIGURAÇÕES A SEREM PROCESSADAS")
    print("=" * 70)
    for algo, mu in configs:
        print(f"   {algo.upper()} - μ = {mu}")
    
    # Processa todos os arquivos
    all_results = compare_all_files(base_path, configs)
    
    # ============================================
    # GERAR TABELAS DE RESULTADOS
    # ============================================
    
    if all_results:
        # Cria diretório de saída
        output_dir = "Resultados/spearman_ale_ranking"
        os.makedirs(output_dir, exist_ok=True)
        
        # 1. Tabela resumo
        print("\n" + "=" * 70)
        print("📊 TABELA RESUMO")
        print("=" * 70)
        
        df_summary = create_summary_table(all_results)
        print(df_summary.to_string(index=False))
        df_summary.to_csv(os.path.join(output_dir, "spearman_summary.csv"), index=False)
        print(f"\n💾 Salvo: {output_dir}/spearman_summary.csv")
        
        # 2. Tabela comparativa completa
        print("\n" + "=" * 70)
        print("📊 TABELA COMPARATIVA COMPLETA")
        print("=" * 70)
        
        df_comp = create_complete_table(all_results)
        pd.set_option('display.max_colwidth', 60)
        print(df_comp.to_string(index=False))
        df_comp.to_csv(os.path.join(output_dir, "spearman_complete_comparison.csv"), index=False)
        print(f"\n💾 Salvo: {output_dir}/spearman_complete_comparison.csv")
        
        # 3. Tabela LaTeX (resumo)
        latex_table = generate_latex_table(df_summary)
        with open(os.path.join(output_dir, "spearman_table.tex"), "w") as f:
            f.write(latex_table)
        print(f"📄 Tabela LaTeX (resumo): {output_dir}/spearman_table.tex")
        
        # 4. Tabela LaTeX (completa)
        latex_table_complete = generate_latex_table_complete(df_comp)
        with open(os.path.join(output_dir, "spearman_table_complete.tex"), "w") as f:
            f.write(latex_table_complete)
        print(f"📄 Tabela LaTeX (completa): {output_dir}/spearman_table_complete.tex")
        
        # ============================================
        # RESULTADO FINAL AGGREGADO
        # ============================================
        
        print("\n" + "=" * 70)
        print("📊 RESULTADO FINAL")
        print("=" * 70)
        
        # Calcula média das correlações
        valid_corrs = [r['result']['spearman'] for r in all_results.values() if not pd.isna(r['result']['spearman'])]
        if valid_corrs:
            mean_corr = np.mean(valid_corrs)
            print(f"   Média das correlações de Spearman: {mean_corr:.3f}")
        
        # Conta status
        status_counts = {}
        for info in all_results.values():
            status = info['result']['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        print("\n   Distribuição dos status:")
        for status, count in status_counts.items():
            print(f"      {status}: {count} configurações")
        
        print("\n" + "=" * 70)
        print("✅ ANÁLISE CONCLUÍDA!")
        print("=" * 70)


📁 CONFIGURAÇÕES A SEREM PROCESSADAS
   LOUVAIN - μ = 0_2
   LOUVAIN - μ = 0_3
   LOUVAIN - μ = 0_4
   LPA - μ = 0_2
   LPA - μ = 0_3
🔍 ANÁLISE DE CORRELAÇÃO DE SPEARMAN: ALE vs RANKING DE REFERÊNCIA

📊 LOUVAIN - μ = 0_2

🏷️  Ranking de Referência (Top 5):
   1. Eigenvector
   2. Triangle Participation
   3. Expansion
   4. Clustering Coefficient
   5. E Out

📈 Ranking ALE (Top 5 por importance_mean):
   1. Eigenvector (0.2048)
   2. Triangle Participation (0.1041)
   3. Expansion (0.0934)
   4. Clustering Coefficient (0.0743)
   5. Shortest Path (0.0658)

📊 Correlação de Spearman:
   Coeficiente: 1.000
   p-valor: 0.0000
   Features em comum: 4/5
   Status: ✅ Alta concordância

📊 LOUVAIN - μ = 0_3

🏷️  Ranking de Referência (Top 5):
   1. Eigenvector
   2. Triangle Participation
   3. Expansion
   4. Clustering Coefficient
   5. E Out

📈 Ranking ALE (Top 5 por importance_mean):
   1. Expansion (0.0956)
   2. Eigenvector (0.0792)
   3. Triangle Participation (0.0599)
   4. Clustering C